# 📊 Complete Guide to ColumnTransformer

## 🎯 What is ColumnTransformer?

`ColumnTransformer` allows you to apply different preprocessing steps to different columns of a dataset.

It is essential when:
- You have mixed data types (numeric + categorical)
- Different columns require different preprocessing
- You want to integrate preprocessing inside a Pipeline
- You want to prevent data leakage

---

## 🔑 Why Use It?

- Selective transformations per column
- Clean integration with `Pipeline`
- Prevents leakage during cross-validation
- Production-ready preprocessing

In [ ]:
# Basci syntax of ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

column_transformer = ColumnTransformer(
    transformers=[
        ('name1', StandardScaler(), ['col1', 'col2']),
        ('name2', OneHotEncoder(), ['col3'])
    ],
    remainder='drop'  # or 'passthrough'
)

## 🔑 Key Parameters of ColumnTransformer

| Parameter | Options | Description |
|------------|----------|-------------|
| `transformers` | list of tuples | Each tuple defines a transformation in the format: `(name, transformer, columns)` |
| `remainder` | `'drop'`, `'passthrough'`, estimator | Specifies how to handle columns not listed in `transformers` |
| `sparse_threshold` | float (default = 0.3) | If the output contains sparse matrices, they will be stacked as sparse only if the density is below this threshold |
| `verbose_feature_names_out` | bool (default = True) | If True, adds the transformer name as a prefix to output feature names |
| `transformer_weights` | dict | Assigns multiplicative weights to features from specific transformers |

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [3]:
df = pd.read_csv('data/covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [5]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [6]:
df['city'].value_counts()   

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [7]:
df.isna().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('has_covid', axis=1), df['has_covid'], test_size=0.2, random_state=42)

In [9]:
X_train.head()

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi


In [12]:
X_train.shape

(80, 5)

### Aam jindagi

In [ ]:
# SimpleImputer --> Fever
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_Test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [ ]:
# OrdinalEncoder --> Cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape



(80, 1)

In [ ]:
# OneHotEncoder --> Gender  City
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])
X_train_gender_city.shape

(80, 4)

In [20]:
X_train_gender_city[:5]

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [1., 1., 0., 0.],
       [0., 1., 0., 0.]])

In [21]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_train_age.shape

(80, 1)

In [25]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age, X_Test_fever, X_test_gender_city, X_test_cough), axis=1)   
X_train_transformed.shape

(80, 7)

### Mentos Jindagi

In [26]:
from sklearn.compose import ColumnTransformer


In [27]:
transformer = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city']),
    ('ordinal', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('imputer', SimpleImputer(), ['fever'])
], remainder='passthrough')

In [28]:
transformer.fit_transform(X_train).shape

(80, 7)

In [30]:
transformer.fit_transform(X_test).shape

(20, 7)

In [32]:
transformer.fit_transform(X_train)

array([[  0.,   0.,   0.,   1.,   0., 101.,  81.],
       [  0.,   0.,   1.,   0.,   0., 100.,   5.],
       [  0.,   0.,   1.,   0.,   0., 100.,  19.],
       [  1.,   1.,   0.,   0.,   0., 100.,  27.],
       [  0.,   1.,   0.,   0.,   0., 103.,  73.],
       [  1.,   0.,   1.,   0.,   1., 103.,  70.],
       [  0.,   1.,   0.,   0.,   0., 102.,  49.],
       [  0.,   0.,   1.,   0.,   1., 101.,  51.],
       [  0.,   1.,   0.,   0.,   0., 101.,  64.],
       [  0.,   0.,   1.,   0.,   0., 101.,  83.],
       [  0.,   0.,   0.,   1.,   0.,  98.,  65.],
       [  0.,   0.,   0.,   0.,   0., 104.,  18.],
       [  0.,   0.,   0.,   0.,   0., 103.,  16.],
       [  1.,   0.,   1.,   0.,   0., 104.,  16.],
       [  1.,   0.,   1.,   0.,   0., 100.,  27.],
       [  0.,   0.,   0.,   0.,   0., 101.,  84.],
       [  1.,   0.,   1.,   0.,   0., 104.,  51.],
       [  0.,   0.,   0.,   0.,   0., 102.,  69.],
       [  0.,   0.,   0.,   0.,   1., 102.,  82.],
       [  0.,   0.,   1.,   0.,